# 04b - Generalisation / Leakage Check

**Why this notebook exists.** In notebook 04, Random Forest scored almost perfectly (AUPRC 0.998, precision 1.000). In security that is a *warning sign*, not a trophy.

Each IoT-23 file is one malware capture. Our earlier split was **random**, so rows from the *same* capture appeared in both training and test. A model can then 'cheat' by learning fingerprints of a specific capture (a particular port, a byte-size quirk) instead of learning real attack behaviour - and still ace a test full of those same captures.

**The honest test:** hold out *whole capture files* for testing, so the model is judged only on captures it has never seen. We call this a **grouped split** (group = the file).

- If the scores stay high -> the model really generalises; 0.998 is trustworthy.
- If the scores drop -> there was leakage; the random-split numbers were over-optimistic. That is an important, markable finding - not a failure.

## Setup

In [ ]:
import os, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay, precision_score,
                             recall_score, f1_score, average_precision_score,
                             PrecisionRecallDisplay)

DATA_ROOT = r"C:\Users\Asus\OneDrive\Desktop\MSc Cybersecurity -NTU\Major Project\Dataset\iot_23_datasets_small"
PROJECT_ROOT = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
FIG_DIR = os.path.join(PROJECT_ROOT, "outputs", "figures")
os.makedirs(FIG_DIR, exist_ok=True)

## Step 1 - Load the data, remembering which file each row came from

Same loader as before, with one addition: we record a `source_file` for every row. That is the **group** we will split on.

In [ ]:
ROWS_PER_FILE = 50000
all_files = sorted(glob.glob(DATA_ROOT + "/**/*.labeled", recursive=True))
print(f"Found {len(all_files)} labeled files")

all_chunks = []
for filepath in all_files:
    file_id = os.path.relpath(filepath, DATA_ROOT)   # short unique name per capture
    col_names = []
    with open(filepath) as f:
        for line in f:
            if line.startswith("#fields"):
                col_names = line.strip().split("\t")[1:]
                break
    rows = 0
    for chunk in pd.read_csv(filepath, sep="\t", comment="#", header=None,
                             low_memory=False, chunksize=50000):
        chunk.columns = col_names
        last_col = chunk.columns[-1]
        split = chunk[last_col].str.strip().str.split(r"\s{2,}", expand=True, n=2, regex=True)
        if split.shape[1] == 3:
            split.columns = ["tunnel_parents", "label", "detailed_label"]
            chunk = chunk.drop(columns=[last_col])
            chunk = pd.concat([chunk, split], axis=1)
        chunk["label"] = chunk["label"].str.lower().str.strip()
        chunk["source_file"] = file_id          # <-- the group label
        all_chunks.append(chunk)
        rows += len(chunk)
        if rows >= ROWS_PER_FILE:
            break

df = pd.concat(all_chunks, ignore_index=True)
print(f"Loaded {len(df):,} rows from {df['source_file'].nunique()} files")
print(df["label"].value_counts())

## Step 2 - Build target, features and groups (same preparation as notebook 02)

In [ ]:
# Target: 1 = malicious, 0 = benign
y = (df["label"] == "malicious").astype(int).to_numpy()
groups = df["source_file"].to_numpy()

categorical_features = ["proto", "service", "conn_state"]
numeric_features = ["duration", "orig_bytes", "resp_bytes", "missed_bytes",
                    "orig_pkts", "resp_pkts", "orig_ip_bytes", "resp_ip_bytes", "id.resp_p"]

X = df[categorical_features + numeric_features + ["history"]].copy()

# Clean (identical to notebook 02)
for col in numeric_features:
    X[col] = pd.to_numeric(X[col], errors="coerce").fillna(0)
for col in categorical_features:
    X[col] = X[col].replace("-", "unknown").fillna("unknown")

hist = X["history"].fillna("").str.lower()
X["has_syn"] = hist.str.contains("s").astype(int)
X["has_ack"] = hist.str.contains("a").astype(int)
X["has_fin"] = hist.str.contains("f").astype(int)
X["has_rst"] = hist.str.contains("r").astype(int)
flag_features = ["has_syn", "has_ack", "has_fin", "has_rst"]
X = X.drop(columns=["history"])
print("Feature table ready:", X.shape)

## Step 3 - Grouped split: hold out whole files for testing

`GroupShuffleSplit` guarantees that no file appears in both train and test. Because some IoT-23 files are all-benign and others all-attack, a careless split could leave the test set with only one class. So we try a few random seeds and keep the first split where **both classes are present, in reasonable numbers, on both sides.** We print which files went where.

In [ ]:
def find_grouped_split(X, y, groups, test_size=0.3, min_per_class=50, tries=300):
    for seed in range(tries):
        gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)
        tr, te = next(gss.split(X, y, groups))
        ok = (y[tr].sum() >= min_per_class and (len(tr) - y[tr].sum()) >= min_per_class and
              y[te].sum() >= min_per_class and (len(te) - y[te].sum()) >= min_per_class)
        if ok:
            return tr, te, seed
    return None, None, None

train_idx, test_idx, seed = find_grouped_split(X, y, groups)
if train_idx is None:
    print("Could not find a clean grouped split (a class may live in too few files).")
    print("Benign appears in files:", sorted(set(groups[y == 0])))
    print("This itself is a finding - tell Claude and we will adapt the approach.")
else:
    print(f"Found a clean grouped split (seed={seed}).")
    print(f"Train rows: {len(train_idx):,} | Test rows: {len(test_idx):,}")
    print(f"Train malicious %: {y[train_idx].mean():.1%} | Test malicious %: {y[test_idx].mean():.1%}")
    print("\nTRAIN files:", sorted(set(groups[train_idx])))
    print("\nTEST files (unseen):", sorted(set(groups[test_idx])))

## Step 4 - Fit preprocessing on TRAIN files only, then train both models

Exactly the same preprocessing as notebook 02 (RobustScaler + one-hot + passthrough flags), but fitted only on the training files and applied to the unseen test files.

In [ ]:
X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
y_tr, y_te = y[train_idx], y[test_idx]

preprocessor = ColumnTransformer([
    ("num", RobustScaler(), numeric_features),
    ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=0.01, sparse_output=False),
     categorical_features),
    ("flags", "passthrough", flag_features),
], remainder="drop")

X_tr_p = preprocessor.fit_transform(X_tr)
X_te_p = preprocessor.transform(X_te)

logreg = LogisticRegression(class_weight="balanced", max_iter=1000).fit(X_tr_p, y_tr)
rf = RandomForestClassifier(n_estimators=100, class_weight="balanced",
                            n_jobs=-1, random_state=42).fit(X_tr_p, y_tr)
print("Both models retrained on the grouped (held-out-files) training set.")

## Step 5 - Evaluate on the unseen files and compare to the random split

The reference numbers below are your notebook-04 results (random split). The new column is performance on captures the model never saw. **Compare them.** A big drop in AUPRC/recall = leakage was inflating the random-split scores.

In [ ]:
random_split_reference = {
    "Logistic Regression": {"Precision": 0.924, "Recall": 0.857, "F1": 0.889, "AUPRC": 0.970},
    "Random Forest":       {"Precision": 1.000, "Recall": 0.947, "F1": 0.973, "AUPRC": 0.998},
}

rows = []
for name, model in {"Logistic Regression": logreg, "Random Forest": rf}.items():
    y_pred = model.predict(X_te_p)
    y_score = model.predict_proba(X_te_p)[:, 1]
    ref = random_split_reference[name]
    rows.append({
        "Model": name,
        "Precision (grouped)": round(precision_score(y_te, y_pred, zero_division=0), 3),
        "Recall (grouped)":    round(recall_score(y_te, y_pred, zero_division=0), 3),
        "F1 (grouped)":        round(f1_score(y_te, y_pred, zero_division=0), 3),
        "AUPRC (grouped)":     round(average_precision_score(y_te, y_score), 3),
        "AUPRC (random ref)":  ref["AUPRC"],
        "Recall (random ref)": ref["Recall"],
    })

comparison = pd.DataFrame(rows).set_index("Model")
print(comparison.to_string())
comparison.to_csv(os.path.join(PROJECT_ROOT, "outputs", "generalisation_comparison.csv"))
print("\nSaved to outputs/generalisation_comparison.csv")

In [ ]:
# Confusion matrix for Random Forest on the unseen files
y_pred_rf = rf.predict(X_te_p)
cm = confusion_matrix(y_te, y_pred_rf)
ConfusionMatrixDisplay(cm, display_labels=["benign", "malicious"]).plot(colorbar=False)
plt.title("Random Forest on UNSEEN capture files")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "rf_grouped_confusion.png"), dpi=150, bbox_inches="tight")
plt.show()

## How to read this (for your report)

Write, in your own words, the comparison between the random split and the grouped split:

- If AUPRC/recall held up: the model generalises across captures - state that the result is robust and not an artefact of leakage.
- If they dropped: explain that the random split over-stated performance because of capture-level leakage, that the grouped split is the fairer estimate, and what that means for real deployment. Discuss which features likely caused it (SHAP in notebook 05 will help confirm).

Either way, *doing this check* is exactly the critical, honest evaluation the marking scheme rewards.